# Lab 09 — Knowledge Distillation with a Local DeepSeek Teacher

**Goal:** transfer behavior from a much stronger local teacher model into our fixed `Qwen/Qwen3-0.6B` student.

Teacher:

```text
oMLX
  ↓
DeepSeek-V4-Flash-0731-MXFP4-MLX
```

Student:

```text
Qwen3-0.6B
```

This lab introduces **response distillation**:

```text
teacher sees task/state
        ↓
teacher produces target response
        ↓
student is trained on teacher response
```

This differs from Lab 06/07, where our hard labels came directly from the symbolic expert.

We will also distinguish response distillation from **soft-target/logit distillation**, but the primary experiment will not depend on oMLX logprobs because current oMLX chat-completion logprob support is unreliable.

The point is not to perfect Wordle. The point is to learn what teacher→student transfer actually looks like.

## 9.1 Architecture of the experiment

We now have three sources of behavior:

```text
Symbolic expert
    │
    │ exact Wordle semantics / reference
    ▼

DeepSeek teacher
    │
    │ natural-language / policy behavior
    ▼

Qwen3-0.6B student
```

The symbolic expert remains useful even though it is not the distillation teacher. It lets us check whether DeepSeek itself gives sane Wordle answers before we train Qwen to imitate it.

Distilling a bad teacher is just automated error propagation.

## 9.2 Prerequisites

Your oMLX server should already be running.

Default API base:

```text
http://localhost:8000/v1
```

Install the OpenAI client if needed:

```bash
pip install -U openai
```

This notebook uses oMLX through its OpenAI-compatible API.

In [2]:
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, load_dataset
from openai import OpenAI
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("device:", device)
print("PyTorch:", torch.__version__)

device: mps
PyTorch: 2.13.0


## 9.3 Connect to oMLX and discover the teacher model ID

Do not hard-code the exact served model ID until we see what oMLX reports.

oMLX exposes `GET /v1/models`, so we discover it directly.

In [3]:
OMLX_BASE_URL = "http://localhost:8000/v1"

teacher_client = OpenAI(
    base_url=OMLX_BASE_URL,
    api_key="not-needed",
)

models = teacher_client.models.list()

print("Models served by oMLX:")
for item in models.data:
    print(" -", item.id)

Models served by oMLX:
 - DeepSeek-V4-Flash-0731-MXFP4-MLX
 - Falcon-OCR-bf16
 - Nemotron-3-Nano-Omni-30B-A3B-Reasoning-bf16
 - Ornith-1.0-35B-bf16
 - PaddleOCR-VL-1.5-bf16
 - Qwen3.6-35B-A3B-DFlash
 - Qwen3.6-35B-A3B-bf16
 - Qwen3.8-27B
 - Qwen3.8-27B-bf16
 - Qwopus3.6-35B-A3B-Coder-bf16
 - ThinkingCap-Qwen3.6-27B-OptiQ-4bit
 - chandra-ocr-2-oQ8
 - dots.mocr-bf16
 - gemma-4-12B-it-8bit
 - gemma-4-31B-it-DFlash
 - gemma-4-31b-it-5bit
 - gemma-4-31b-it-bf16
 - mikoy92--Unlimited-OCR-bf16-mlx
 - mlx-community--DeepSeek-OCR-2-bf16
 - mlx-community--GLM-OCR-bf16
 - mlx-community--gemma-4-e4b-it-qat-OptiQ-4bit
 - mlx-community--typhoon-ocr1.5-2b-8bit
 - olmOCR-2-7B-1025-bf16


Set `TEACHER_MODEL` to the exact DeepSeek model ID printed above.

If only one model is served, the helper below will choose it automatically.

In [4]:
model_ids = [item.id for item in models.data]

deepseek_ids = [
    model_id
    for model_id in model_ids
    if "deepseek" in model_id.lower()
]

if len(deepseek_ids) == 1:
    TEACHER_MODEL = deepseek_ids[0]
elif len(model_ids) == 1:
    TEACHER_MODEL = model_ids[0]
else:
    TEACHER_MODEL = "DeepSeek-V4-Flash-0731-MXFP4-MLX"

print("auto-selected teacher:", TEACHER_MODEL)
print()
print(
    "If this is None or wrong, set TEACHER_MODEL manually "
    "to the exact DeepSeek ID above."
)

auto-selected teacher: DeepSeek-V4-Flash-0731-MXFP4-MLX

If this is None or wrong, set TEACHER_MODEL manually to the exact DeepSeek ID above.


## 9.4 Smoke-test the teacher

We want a deterministic teacher for dataset generation.

Use temperature `0`.

In [5]:
assert TEACHER_MODEL is not None, "Set TEACHER_MODEL first."

def teacher_chat(
    prompt: str,
    max_tokens: int = 64,
) -> str:
    response = teacher_client.chat.completions.create(
        model=TEACHER_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0,
        max_tokens=max_tokens,
    )

    return (
        response
        .choices[0]
        .message
        .content
        .strip()
    )

smoke = teacher_chat(
    "Return exactly one uppercase five-letter Wordle guess. "
    "Output only the word."
)

print(repr(smoke))

'CRANE'


## 9.5 Load the Lab 06 curriculum

We reuse the persisted split files.

Teacher generation will operate on **train** rows only.

Validation remains untouched so we can compare the distilled student against the earlier SFT and LoRA runs.

In [6]:
DATA_DIR = Path("../data/generated")

dataset = load_dataset(
    "json",
    data_files={
        "train": str(DATA_DIR / "wordle-sft-train.jsonl"),
        "validation": str(DATA_DIR / "wordle-sft-dev.jsonl"),
        "test": str(DATA_DIR / "wordle-sft-test.jsonl"),
    },
)

print(dataset)

assert dataset["train"].num_rows == 16465
assert dataset["validation"].num_rows == 2169
assert dataset["test"].num_rows == 190

DatasetDict({
    train: Dataset({
        features: ['task', 'split', 'answer', 'turn', 'candidate_count', 'prompt', 'response'],
        num_rows: 16465
    })
    validation: Dataset({
        features: ['task', 'split', 'answer', 'turn', 'candidate_count', 'prompt', 'response'],
        num_rows: 2169
    })
    test: Dataset({
        features: ['task', 'split', 'answer', 'turn', 'candidate_count', 'prompt', 'response'],
        num_rows: 190
    })
})


## 9.6 Benchmark the teacher before distilling it

Start with the same three task types:

- `VALID_CANDIDATE`
- `CHOOSE_VALID`
- `NEXT_GUESS`

We evaluate a small deterministic sample first.

The teacher may not reproduce the symbolic expert's **exact** `NEXT_GUESS` word even when its answer is perfectly reasonable. Therefore:

- exact accuracy is meaningful for `VALID_CANDIDATE`;
- exact accuracy is meaningful for `CHOOSE_VALID`;
- exact accuracy on `NEXT_GUESS` measures imitation of our symbolic expert, not absolute Wordle quality.

Keep that distinction clear.

### Inspect raw reasoning-model responses

Before running the benchmark or generating teacher labels, inspect one raw response for each task. Reasoning models may expose the final answer and reasoning in different fields, so a zero score can indicate an output-parsing problem rather than a teacher-capability problem.

**Stop after this cell and inspect all three responses before continuing.**

In [7]:
for task in [
    "VALID_CANDIDATE",
    "CHOOSE_VALID",
    "NEXT_GUESS",
]:
    row = next(
        item
        for item in dataset["validation"]
        if item["task"] == task
    )

    response = teacher_client.chat.completions.create(
        model=TEACHER_MODEL,
        messages=[
            {
                "role": "user",
                "content": row["prompt"],
            }
        ],
        temperature=0,
        max_tokens=128,
    )

    message = response.choices[0].message

    print(f"\n--- {task} ---")
    print("TARGET:", row["response"])
    print("CONTENT:", repr(message.content))
    print("MESSAGE:", message)

    reasoning = getattr(message, "reasoning_content", None)
    if reasoning is not None:
        print("REASONING:", repr(reasoning))


--- VALID_CANDIDATE ---
TARGET: VALID
CONTENT: 'We need answer exactly VALID or INVALID. Need solve Wordle constraints. Need determine if candidate L E A F Y could be hidden answer given history.\n\nWe need parse history. Wordle guesses and feedback colors: B = gray (not in word), Y = yellow (in word wrong position), G? Here B/Y? Actually "R A I S E -> B Y B B Y" means guess RAISE, feedback: R gray, A yellow, I gray, S gray, E yellow. So for RAISE: R not in answer, A is in answer but not position 2? Wait positions: R pos'
MESSAGE: ChatCompletionMessage(content='We need answer exactly VALID or INVALID. Need solve Wordle constraints. Need determine if candidate L E A F Y could be hidden answer given history.\n\nWe need parse history. Wordle guesses and feedback colors: B = gray (not in word), Y = yellow (in word wrong position), G? Here B/Y? Actually "R A I S E -> B Y B B Y" means guess RAISE, feedback: R gray, A yellow, I gray, S gray, E yellow. So for RAISE: R not in answer, A is in a

In [8]:
def normalize_first_line(text: str) -> str:
    return (
        text
        .strip()
        .splitlines()[0]
        .strip()
        .strip("*`")
        .upper()
    )

def teacher_answer_row(row: dict) -> str:
    return normalize_first_line(
        teacher_chat(
            row["prompt"],
            max_tokens=32,
        )
    )

TEACHER_EVAL_PER_TASK = 50

teacher_eval_rows = []

for task in [
    "VALID_CANDIDATE",
    "CHOOSE_VALID",
    "NEXT_GUESS",
]:
    task_rows = [
        row
        for row in dataset["validation"]
        if row["task"] == task
    ][:TEACHER_EVAL_PER_TASK]

    for row in task_rows:
        actual = teacher_answer_row(row)
        expected = row["response"].strip().upper()

        teacher_eval_rows.append({
            "task": task,
            "expected": expected,
            "teacher": actual,
            "exact": actual == expected,
        })

teacher_eval_df = pd.DataFrame(teacher_eval_rows)

print("TEACHER EXACT ACCURACY")
print(
    teacher_eval_df
    .groupby("task")["exact"]
    .mean()
)

TEACHER EXACT ACCURACY
task
CHOOSE_VALID       0.0
NEXT_GUESS         0.0
VALID_CANDIDATE    0.0
Name: exact, dtype: float64


## 9.7 Inspect teacher disagreements

Before using the teacher as a label source, inspect disagreements.

For `VALID_CANDIDATE` and `CHOOSE_VALID`, disagreements usually mean the teacher is wrong or failed the output contract.

For `NEXT_GUESS`, a different five-letter word may still be a valid move.

In [ ]:
teacher_disagreements = teacher_eval_df.loc[
    ~teacher_eval_df["exact"]
]

print("teacher disagreements:", len(teacher_disagreements))
teacher_disagreements.head(30)

## 9.8 Teacher validity on NEXT_GUESS

We can evaluate a teacher Wordle action more fairly than exact-match by asking:

> Is the generated five-letter word consistent with the observed history?

Use the trusted Lab 03 scorer.

This still does not measure optimal strategy, but it distinguishes a sane legal state-dependent action from nonsense.

In [ ]:
from tiny_wordle.game import Turn, score_string

WORD_RE = __import__("re").compile(r"^[A-Z]{5}$")

def parse_spaced_history(prompt: str) -> list[Turn]:
    history = []

    if "History:\n" not in prompt:
        return history

    history_text = prompt.split("History:\n", 1)[1]

    # Stop before task-specific option/candidate sections if present.
    for marker in ["\n\nCandidate:", "\n\nOption A:"]:
        history_text = history_text.split(marker, 1)[0]

    if history_text.strip() == "No previous guesses.":
        return []

    for line in history_text.splitlines():
        if "->" not in line:
            continue

        left, right = line.split("->", 1)
        guess = left.replace(" ", "").strip().upper()
        feedback = right.replace(" ", "").strip().upper()

        if len(guess) == 5 and len(feedback) == 5:
            history.append(
                Turn(
                    guess=guess,
                    feedback=feedback,
                )
            )

    return history

def is_history_consistent_guess(
    guess: str,
    history: list[Turn],
) -> bool:
    if not WORD_RE.fullmatch(guess):
        return False

    return all(
        score_string(
            guess,
            old.guess,
        ) == old.feedback
        for old in history
    )

In [ ]:
next_guess_validation_rows = [
    row
    for row in dataset["validation"]
    if row["task"] == "NEXT_GUESS"
][:100]

teacher_next_records = []

for row in next_guess_validation_rows:
    actual = teacher_answer_row(row)
    history = parse_spaced_history(row["prompt"])

    teacher_next_records.append({
        "expected": row["response"],
        "teacher": actual,
        "format_valid": bool(WORD_RE.fullmatch(actual)),
        "history_consistent": (
            is_history_consistent_guess(actual, history)
            if history
            else bool(WORD_RE.fullmatch(actual))
        ),
    })

teacher_next_df = pd.DataFrame(teacher_next_records)

print(
    "teacher NEXT_GUESS format-valid:",
    f"{teacher_next_df['format_valid'].mean():.1%}",
)
print(
    "teacher NEXT_GUESS history-consistent:",
    f"{teacher_next_df['history_consistent'].mean():.1%}",
)

## 9.9 Decide the response-distillation dataset size

Generating all 16,465 training labels through a 300B-class teacher is unnecessary for the first lab.

We want to learn the technique, not turn this into a weekend-long inference job.

Start with a stratified sample:

```text
VALID_CANDIDATE   1,000
CHOOSE_VALID      1,000
NEXT_GUESS        1,000
```

Total: 3,000 teacher-labeled examples.

If this works, scaling teacher data becomes an ordinary follow-up experiment.

In [ ]:
TEACHER_ROWS_PER_TASK = 1000

rng = np.random.default_rng(SEED)

selected_indices = []

for task in [
    "VALID_CANDIDATE",
    "CHOOSE_VALID",
    "NEXT_GUESS",
]:
    task_indices = [
        i
        for i, row in enumerate(dataset["train"])
        if row["task"] == task
    ]

    take = min(
        TEACHER_ROWS_PER_TASK,
        len(task_indices),
    )

    chosen = rng.choice(
        task_indices,
        size=take,
        replace=False,
    )

    selected_indices.extend(
        int(x)
        for x in chosen
    )

selected_indices = sorted(selected_indices)

print("teacher-generation rows:", len(selected_indices))

## 9.10 Generate teacher responses and cache them

Teacher inference is the expensive part.

Cache every result immediately so rerunning later cells does not regenerate completed responses.

If the notebook stops midway, rerun this cell; already-cached rows will be skipped.

In [ ]:
DISTILL_DIR = DATA_DIR / "distillation"
DISTILL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TEACHER_CACHE = (
    DISTILL_DIR
    / "deepseek_teacher_responses.jsonl"
)

cached = {}

if TEACHER_CACHE.exists():
    for line in TEACHER_CACHE.read_text().splitlines():
        if not line.strip():
            continue

        record = json.loads(line)
        cached[int(record["source_index"])] = record

print("cached teacher rows:", len(cached))

In [ ]:
start = time.perf_counter()

for position, source_index in enumerate(
    selected_indices,
    1,
):
    if source_index in cached:
        continue

    row = dataset["train"][source_index]

    teacher_raw = teacher_chat(
        row["prompt"],
        max_tokens=32,
    )

    teacher_response = normalize_first_line(
        teacher_raw
    )

    record = {
        "source_index": source_index,
        "task": row["task"],
        "answer": row["answer"],
        "turn": row["turn"],
        "candidate_count": row["candidate_count"],
        "prompt": row["prompt"],
        "symbolic_response": row["response"],
        "teacher_raw": teacher_raw,
        "teacher_response": teacher_response,
    }

    with TEACHER_CACHE.open("a") as f:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

    cached[source_index] = record

    if position == 1 or position % 50 == 0:
        elapsed = time.perf_counter() - start
        print(
            f"{position}/{len(selected_indices)} | "
            f"cached={len(cached)} | "
            f"{elapsed:.1f}s"
        )

print("teacher generation complete")
print("cached rows:", len(cached))

## 9.11 Inspect teacher-label quality

Do not blindly train on raw teacher text.

For this first response-distillation lab, keep only outputs that obey the task's output contract:

```text
VALID_CANDIDATE → VALID or INVALID
CHOOSE_VALID    → uppercase five-letter word
NEXT_GUESS      → uppercase five-letter word
```

This filters prose and malformed generations.

In [ ]:
teacher_records = [
    cached[i]
    for i in selected_indices
    if i in cached
]

teacher_df = pd.DataFrame(
    teacher_records
)

def teacher_contract_valid(row) -> bool:
    response = row["teacher_response"]

    if row["task"] == "VALID_CANDIDATE":
        return response in {
            "VALID",
            "INVALID",
        }

    return bool(
        WORD_RE.fullmatch(response)
    )

teacher_df["contract_valid"] = (
    teacher_df.apply(
        teacher_contract_valid,
        axis=1,
    )
)

print("teacher rows:", len(teacher_df))
print(
    "contract-valid:",
    f"{teacher_df['contract_valid'].mean():.1%}",
)
print()
print("BY TASK")
print(
    teacher_df
    .groupby("task")["contract_valid"]
    .mean()
)

## 9.12 Compare teacher labels with symbolic labels

This is not about deciding which teacher is universally better.

It tells us how different the teacher-generated supervision actually is.

In [ ]:
teacher_df["same_as_symbolic"] = (
    teacher_df["teacher_response"].str.upper()
    == teacher_df["symbolic_response"].str.upper()
)

print("teacher == symbolic:")
print(
    teacher_df
    .groupby("task")["same_as_symbolic"]
    .mean()
)

For `NEXT_GUESS`, disagreement is expected and potentially useful.

For `VALID_CANDIDATE`, disagreement deserves more suspicion because the symbolic label is exact.

## 9.13 Build the distilled training set

Use only contract-valid teacher responses.

The student will see the **same prompt** and learn the **teacher response** rather than the symbolic response.

This is response distillation.

In [ ]:
distill_df = (
    teacher_df.loc[
        teacher_df["contract_valid"]
    ]
    .copy()
)

distill_df = distill_df[
    [
        "task",
        "answer",
        "turn",
        "candidate_count",
        "prompt",
        "teacher_response",
    ]
].rename(
    columns={
        "teacher_response": "response"
    }
)

print("distillation rows:", len(distill_df))
print()
print(distill_df["task"].value_counts())

## 9.14 Persist the teacher-distillation dataset

In [ ]:
DISTILL_TRAIN_PATH = (
    DISTILL_DIR
    / "wordle-deepseek-distill-train.jsonl"
)

distill_df.to_json(
    DISTILL_TRAIN_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

print(DISTILL_TRAIN_PATH)
print(
    f"{DISTILL_TRAIN_PATH.stat().st_size / 1024:.1f} KiB"
)

## 9.15 Load a fresh Qwen student

We distill into a **fresh base Qwen3-0.6B**.

Do not initialize from Lab 07 or Lab 08. Otherwise we would be measuring continued fine-tuning, not teacher→student transfer from the common base.

In [ ]:
STUDENT_MODEL_ID = "Qwen/Qwen3-0.6B"

student_tokenizer = AutoTokenizer.from_pretrained(
    STUDENT_MODEL_ID
)

student = (
    AutoModelForCausalLM
    .from_pretrained(
        STUDENT_MODEL_ID,
        dtype=torch.float32,
    )
    .to(device)
)

print(type(student).__name__)
print(
    "trainable:",
    f"{sum(p.numel() for p in student.parameters() if p.requires_grad):,}",
)

## 9.16 Student formatting and response-only labels

Same training mechanism as Labs 02, 07, and 08.

Distillation changes **where the targets came from**, not how causal-LM loss works.

In [ ]:
MAX_LENGTH = 128

def student_render_prompt(
    prompt: str,
) -> str:
    return (
        student_tokenizer
        .apply_chat_template(
            [
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    )

def encode_student_example(
    example: dict,
) -> dict:
    prompt_text = student_render_prompt(
        example["prompt"]
    )

    full_text = (
        prompt_text
        + example["response"]
        + student_tokenizer.eos_token
    )

    prompt_ids = student_tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]

    full_ids = student_tokenizer(
        full_text,
        add_special_tokens=False,
    )["input_ids"]

    if len(full_ids) >= MAX_LENGTH:
        raise ValueError(
            f"sequence too long: {len(full_ids)}"
        )

    labels = (
        [-100] * len(prompt_ids)
        + full_ids[len(prompt_ids):]
    )

    return {
        "input_ids": full_ids,
        "labels": labels,
    }

## 9.17 Build DataLoader

Because the distilled dataset is only ~3,000 examples, one epoch is a much smaller experiment than Lab 07.

In [ ]:
distill_dataset = Dataset.from_pandas(
    distill_df,
    preserve_index=False,
)

PAD_ID = student_tokenizer.pad_token_id

if PAD_ID is None:
    PAD_ID = student_tokenizer.eos_token_id

def collate_student(rows):
    encoded = [
        encode_student_example(row)
        for row in rows
    ]

    max_len = max(
        len(x["input_ids"])
        for x in encoded
    )

    input_rows = []
    label_rows = []
    attention_rows = []

    for item in encoded:
        ids = item["input_ids"]
        labels = item["labels"]

        pad = max_len - len(ids)

        input_rows.append(
            ids + [PAD_ID] * pad
        )
        label_rows.append(
            labels + [-100] * pad
        )
        attention_rows.append(
            [1] * len(ids)
            + [0] * pad
        )

    return {
        "input_ids": torch.tensor(
            input_rows,
            dtype=torch.long,
        ),
        "labels": torch.tensor(
            label_rows,
            dtype=torch.long,
        ),
        "attention_mask": torch.tensor(
            attention_rows,
            dtype=torch.long,
        ),
    }

BATCH_SIZE = 16

distill_loader = DataLoader(
    distill_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_student,
    generator=torch.Generator().manual_seed(
        SEED
    ),
)

print(
    "distillation rows:",
    len(distill_dataset),
)
print(
    "training batches:",
    len(distill_loader),
)

## 9.18 Train the distilled student

Use full-parameter fine-tuning again.

Why?

Because the technique under study is **distillation**, not LoRA.

Changing both the label source and parameterization would make the comparison muddy.

Use one epoch with the same conservative full-SFT learning rate from Lab 07.

In [ ]:
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 1

optimizer = AdamW(
    student.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_steps = (
    len(distill_loader)
    * EPOCHS
)

warmup_steps = max(
    1,
    int(total_steps * 0.05),
)

def lr_multiplier(step: int) -> float:
    if step < warmup_steps:
        return (
            (step + 1)
            / warmup_steps
        )

    progress = (
        (step - warmup_steps)
        / max(
            1,
            total_steps - warmup_steps,
        )
    )

    return 0.5 * (
        1.0
        + math.cos(
            math.pi * progress
        )
    )

scheduler = (
    torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lr_multiplier,
    )
)

print("total steps:", total_steps)
print("warmup steps:", warmup_steps)

In [ ]:
STUDENT_CHECKPOINT = Path(
    "../checkpoints/"
    "qwen3-0.6b-deepseek-distilled"
)

STUDENT_CHECKPOINT.mkdir(
    parents=True,
    exist_ok=True,
)

student.train()

history = []
start = time.perf_counter()

for step, batch in enumerate(
    distill_loader,
    1,
):
    batch = {
        key: value.to(device)
        for key, value in batch.items()
    }

    optimizer.zero_grad(
        set_to_none=True
    )

    outputs = student(**batch)
    loss = outputs.loss

    loss.backward()

    grad_norm = (
        torch.nn.utils.clip_grad_norm_(
            student.parameters(),
            1.0,
        )
    )

    optimizer.step()
    scheduler.step()

    value = (
        loss
        .detach()
        .float()
        .cpu()
        .item()
    )

    history.append({
        "step": step,
        "loss": value,
        "grad_norm": float(grad_norm),
        "lr": scheduler.get_last_lr()[0],
    })

    if (
        step == 1
        or step % 25 == 0
        or step == total_steps
    ):
        elapsed = (
            time.perf_counter()
            - start
        )

        print(
            f"step {step:4d}/{total_steps} | "
            f"loss {value:.4f} | "
            f"grad {float(grad_norm):.3f} | "
            f"lr {scheduler.get_last_lr()[0]:.2e} | "
            f"{elapsed:.1f}s"
        )

elapsed = time.perf_counter() - start

student.save_pretrained(
    STUDENT_CHECKPOINT,
    safe_serialization=True,
)

student_tokenizer.save_pretrained(
    STUDENT_CHECKPOINT
)

print()
print("training seconds:", elapsed)
print("saved:", STUDENT_CHECKPOINT)

## 9.19 Evaluate the distilled student on the untouched validation set

This is intentionally the **original Lab 06 validation set**, not teacher-generated validation data.

That lets us compare the distilled student against:

```text
Full SFT
LoRA
Distilled student
```

on the same evaluation rows.

In [ ]:
def student_generate(
    prompt: str,
    max_new_tokens: int = 16,
) -> str:
    text = student_render_prompt(prompt)

    batch = student_tokenizer(
        text,
        return_tensors="pt",
    ).to(device)

    student.eval()

    with torch.no_grad():
        output = student.generate(
            **batch,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new = output[
        0,
        batch["input_ids"].shape[1]:,
    ]

    result = (
        student_tokenizer
        .decode(
            new,
            skip_special_tokens=True,
        )
        .strip()
    )

    student.train()

    return normalize_first_line(result)

In [ ]:
student_records = []

for i, row in enumerate(
    dataset["validation"],
    1,
):
    actual = student_generate(
        row["prompt"],
    )

    expected = (
        row["response"]
        .strip()
        .upper()
    )

    student_records.append({
        "task": row["task"],
        "expected": expected,
        "student": actual,
        "exact": actual == expected,
    })

    if i % 250 == 0:
        print(
            f"{i}/{dataset['validation'].num_rows}"
        )

student_eval_df = pd.DataFrame(
    student_records
)

print()
print("DISTILLED STUDENT OVERALL")
print(
    f"{student_eval_df['exact'].mean():.1%}"
)

print()
print("BY TASK")
print(
    student_eval_df
    .groupby("task")["exact"]
    .mean()
)

## 9.20 Compare all supervised methods

Fill this table with the actual distilled-student result:

| Metric | Full SFT | LoRA | DeepSeek distill |
|---|---:|---:|---:|
| Training targets | symbolic | symbolic | DeepSeek |
| Training rows | 16,465 | 16,465 | ~3,000 |
| Trainable params | 596M | 2.29M | 596M |
| Overall exact acc | 65.1% | 61.9% | ? |
| VALID_CANDIDATE | 79.8% | 76.3% | ? |
| CHOOSE_VALID | 76.7% | 71.9% | ? |
| NEXT_GUESS | 1.8% | 2.1% | ? |

Do not expect the distilled model to automatically beat the symbolic-label SFT.

It sees much less data and the teacher can disagree with exact symbolic labels.

The lesson is the transfer mechanism and the behavior of teacher-generated supervision.

## 9.21 Optional: run the frozen Lab 04 gameplay benchmark

Point the Lab 04 model loader at:

```text
../checkpoints/qwen3-0.6b-deepseek-distilled
```

Keep everything else frozen.

Record:

```text
solve rate
history consistency
repeat guesses
valid output rate
```

Again, we are not tuning Wordle to perfection. This simply tells us how response distillation transferred into deployment behavior.

# 9.22 What about true logit distillation?

Response distillation gives the student one target sequence:

```text
teacher → "PLANT"
```

True soft-target distillation instead teaches a **distribution**.

Conceptually:

```text
teacher:
PLANT  0.55
SLANT  0.21
PLANK  0.11
BLANK  0.05
...

student:
PLANT  0.20
SLANT  0.08
...

loss = KL(
    teacher_distribution,
    student_distribution
)
```

A common temperature-scaled objective is:

```text
L = α × hard_label_loss
    + (1 - α) × T² × KL(
        softmax(teacher_logits / T),
        softmax(student_logits / T)
      )
```

The teacher's "dark knowledge" is the relative probability it assigns to alternatives, not merely the top answer.

We are not implementing this through oMLX chat logprobs in the primary lab because current oMLX chat-completion logprob responses can be empty.

There is also a second complication:

> DeepSeek and Qwen use different tokenizers.

So comparing raw token-vocabulary logits directly would not be semantically aligned.

## 9.23 How we would do soft distillation correctly for Wordle

Wordle gives us an elegant shared action space:

```text
2,315 possible answer words
```

Instead of comparing DeepSeek token logits with Qwen token logits, define both models over the same Wordle action vocabulary:

```text
P_teacher(word | state)
P_student(word | state)
```

Then distill across those 2,315 actions.

Possible future implementation:

1. choose a game state;
2. score every candidate word under the teacher;
3. normalize scores into a teacher distribution;
4. score the same candidate list under Qwen;
5. minimize KL divergence.

That is closer to classical knowledge distillation than teacher-response SFT.

It is intentionally left as a follow-up because obtaining well-calibrated teacher scores through the current oMLX API is a separate engineering problem.

# Lab 09 checkpoint

Send me these results in order:

1. the model IDs reported by `/v1/models`;
2. selected `TEACHER_MODEL`;
3. teacher smoke-test response;
4. teacher exact accuracy by task on the small benchmark;
5. teacher `NEXT_GUESS` format-valid and history-consistency rates;
6. teacher-generation row count;
7. generation progress/time;
8. teacher contract-valid rate by task;
9. teacher-vs-symbolic agreement by task;
10. final distillation dataset size;
11. distilled-student training loss/time;
12. distilled-student exact accuracy overall and by task;
13. optional frozen Lab 04 gameplay result.

Then we move to **reinforcement learning**.